# sclab in the browser

This notebook runs entirely in your browser: Python is [Pyodide](https://pyodide.org) and nothing is sent to a server.
Run the cells in order (Shift+Enter). The first cell downloads sclab and its dependencies (about a minute the first time; the browser caches them).

Differences from a regular Jupyter kernel:

- There are no threads, so a step runs to completion before the notebook responds again (the dashboard's *Run in background* option is off).
- sclab uses its own ports of the scanpy functions (QC, Preprocess, PCA, Neighbors, Cluster, UMAP, differential expression, scrublet). scanorama, doubletdetection and the R methods are not available.
- Up to about 10,000 to 20,000 cells is comfortable.

In [ ]:
import piplite  # micropip that looks in this site's own wheels (pypi/) first

# anndata needs typing-extensions>=4.16, newer than Pyodide's
await piplite.install("typing-extensions>=4.16", reinstall=True)
await piplite.install("sclab>=0.5.0.dev0")  # 0.5 is the first release that runs in Pyodide
await piplite.install(["pylimma", "scikit-misc"])  # pseudobulk limma DE; HVG seurat_v3
# optional: edgeR DE and scrublet, without numba and annoy (no WebAssembly builds)
await piplite.install(["statsmodels", "scikit-image"])
await piplite.install(["edgepython", "scrublet"], deps=False)

import sclab

print("sclab", sclab.__version__)
sclab.optional_dependencies();

## Example data

A synthetic count matrix: 3,000 cells and 2,000 genes, six cell types with 40 marker genes each (named `type_<i>_marker<k>`), four samples in two batches, and a batch effect on 10% of the genes. Change `n_cells` to try other sizes.

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp

rng = np.random.default_rng(0)
n_cells, n_genes, n_types, n_markers = 3000, 2000, 6, 40

genes = np.array([f"gene{j:04d}" for j in range(n_genes)], dtype=object)
fold = np.ones((n_types, n_genes))
markers = rng.permutation(n_genes)[: n_types * n_markers].reshape(n_types, n_markers)
for t, idx in enumerate(markers):
    fold[t, idx] = rng.uniform(4, 20, n_markers)
    genes[idx] = [f"type_{t}_marker{k}" for k in range(n_markers)]

cell_type = rng.choice(n_types, n_cells, p=[0.3, 0.25, 0.15, 0.12, 0.1, 0.08])
sample = rng.integers(0, 4, n_cells)
batch = sample // 2  # samples 0, 1 in batch 0; 2, 3 in batch 1
batch_effect = np.ones((2, n_genes))
batch_effect[1, rng.choice(n_genes, n_genes // 10, replace=False)] = rng.lognormal(0, 0.7, n_genes // 10)

base = rng.lognormal(-1.5, 1.5, n_genes)
library_size = rng.lognormal(0, 0.3, n_cells)
mean = base * fold[cell_type] * batch_effect[batch] * library_size[:, None]
counts = rng.poisson(mean * rng.gamma(2.0, 0.5, mean.shape))  # negative binomial

obs = pd.DataFrame(
    {
        "cell_type": pd.Categorical([f"type_{t}" for t in cell_type]),
        "sample": pd.Categorical([f"s{s + 1}" for s in sample]),
        "batch": pd.Categorical([f"b{b + 1}" for b in batch]),
    },
    index=[f"cell{i:05d}" for i in range(n_cells)],
)
X = sp.csr_matrix(counts.astype(np.float32))
adata = ad.AnnData(X, obs=obs, var=pd.DataFrame(index=genes), layers={"counts": X.copy()})
del mean, counts
adata

## The dashboard

Run the steps from the left panel in order: QC, Preprocess, PCA, Neighbors, Cluster, UMAP. Then color the plot by `cell_type` or `batch`, try Integration (cca or harmony on `batch`), Gene Expression, or Differential Expression (wilcoxon, or `pseudobulk_limma` with `sample` as replicates).

In [ ]:
from sclab import SCLabDashboard

dashboard = SCLabDashboard(adata, name="Synthetic demo", counts_layer="counts", batch_key="batch")
# Neighbors' Group By Batch needs bbknn, which has no WebAssembly build
dashboard.processor.steps["Neighbors"].variable_controls["group_by_batch"].value = False
dashboard

## Your own data

Drag an `.h5ad` file into the file browser on the left, then `SCLabDashboard("my_data.h5ad")`, or run `SCLabDashboard()` and pick it in the loader. Files stay in this browser's storage.